### EDA of surface weather data

In [46]:
# Packages
import numpy as np
import pandas as pd
from astropy.time import Time
from astropy.coordinates import get_sun, AltAz, EarthLocation
import astropy.units as u
from datetime import datetime

In [2]:
# Load dataset
df_b = pd.read_excel('../data/bronx_mesonet_weather_data.xlsx')
df_m = pd.read_excel('../data/manhattan_mesonet_weather_data.xlsx')

In [3]:
# Statistical description of float columns
df_b.describe()

,Air Temp at Surface [degC],Relative Humidity [percent],Avg Wind Speed [m/s],Wind Direction [degrees],Solar Flux [W/m^2]
count,169.000000,169.000000,169.000000,169.000000,169.000000
mean,24.789941,54.445562,2.348521,128.479290,387.869822
std,2.590496,11.844493,1.023364,92.281601,262.097914
min,19.300000,39.600000,0.200000,1.000000,12.000000
25%,23.900000,47.400000,1.600000,52.000000,156.000000
50%,25.500000,49.800000,2.200000,128.000000,330.000000
75%,26.700000,56.500000,3.100000,168.000000,618.000000
max,28.400000,88.200000,4.800000,359.000000,960.000000


In [4]:
# Statistical description of float columns
df_m.describe()

,Air Temp at Surface [degC],Relative Humidity [percent],Avg Wind Speed [m/s],Wind Direction [degrees],Solar Flux [W/m^2]
count,169.000000,169.000000,169.000000,169.000000,169.000000
mean,25.198225,49.401775,1.931953,134.863905,380.000000
std,1.696547,6.103468,0.799358,86.477491,262.528479
min,21.300000,39.200000,0.600000,2.000000,10.000000
25%,24.400000,45.900000,1.300000,61.000000,140.000000
50%,25.300000,48.000000,1.900000,144.000000,321.000000
75%,26.500000,51.100000,2.400000,171.000000,620.000000
max,27.900000,66.500000,4.100000,355.000000,840.000000


In [5]:
# Counting NaNs
print(df_b.isna().sum())
print("================================")
print(df_m.isna().sum())

Date / Time                    0
Air Temp at Surface [degC]     0
Relative Humidity [percent]    0
Avg Wind Speed [m/s]           0
Wind Direction [degrees]       0
Solar Flux [W/m^2]             0
dtype: int64
Date / Time                    0
Air Temp at Surface [degC]     0
Relative Humidity [percent]    0
Avg Wind Speed [m/s]           0
Wind Direction [degrees]       0
Solar Flux [W/m^2]             0
dtype: int64


In [25]:
# Convert Date and Time column in datetime objects
df_b[df_b.columns[0]] = pd.to_datetime(df_b[df_b.columns[0]])

/home/scollazo/anaconda3/envs/env-ml/lib/python3.10/site-packages/dateutil/parser/_parser.py:1207: UnknownTimezoneWarning: tzname EDT identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "


In [29]:
# Convert Date and Time column in datetime objects
df_m[df_m.columns[0]] = pd.to_datetime(df_m[df_m.columns[0]])

/home/scollazo/anaconda3/envs/env-ml/lib/python3.10/site-packages/dateutil/parser/_parser.py:1207: UnknownTimezoneWarning: tzname EDT identified but not understood.  Pass `tzinfos` argument in order to correctly return a timezone-aware datetime.  In a future version, this will raise an exception.
  warnings.warn("tzname {tzname} identified but not understood.  "


In [61]:
# UTC difference
utcoffset = -4*u.hour                                 # EDT (Eastern Daylight Time)

# Set time
t = Time(df_m[df_m.columns[0]]) - utcoffset

In [62]:
sun = get_sun(t)

In [63]:
sun

<SkyCoord (GCRS: obstime=['2021-07-24T10:00:00.000000000' '2021-07-24T10:05:00.000000000'
 '2021-07-24T10:10:00.000000000' '2021-07-24T10:15:00.000000000'
 '2021-07-24T10:20:00.000000000' '2021-07-24T10:25:00.000000000'
 '2021-07-24T10:30:00.000000000' '2021-07-24T10:35:00.000000000'
 '2021-07-24T10:40:00.000000000' '2021-07-24T10:45:00.000000000'
 '2021-07-24T10:50:00.000000000' '2021-07-24T10:55:00.000000000'
 '2021-07-24T11:00:00.000000000' '2021-07-24T11:05:00.000000000'
 '2021-07-24T11:10:00.000000000' '2021-07-24T11:15:00.000000000'
 '2021-07-24T11:20:00.000000000' '2021-07-24T11:25:00.000000000'
 '2021-07-24T11:30:00.000000000' '2021-07-24T11:35:00.000000000'
 '2021-07-24T11:40:00.000000000' '2021-07-24T11:45:00.000000000'
 '2021-07-24T11:50:00.000000000' '2021-07-24T11:55:00.000000000'
 '2021-07-24T12:00:00.000000000' '2021-07-24T12:05:00.000000000'
 '2021-07-24T12:10:00.000000000' '2021-07-24T12:15:00.000000000'
 '2021-07-24T12:20:00.000000000' '2021-07-24T12:25:00.000000000'


In [64]:
# Mean latitude
mean_lat = (40.76754 + 40.87248)/2.0                 # degrees

# Mean longitude
mean_lon = (-73.96449 - 73.89352)/2.0                # degrees

# Mean altitude
mean_alt = (94.8 + 57.5)/2.0                         # meters

# Set the location
new_york = EarthLocation(lat = mean_lat*u.deg, lon = mean_lon*u.deg, height = mean_alt*u.m)

In [65]:
local_sun = sun.transform_to(AltAz(obstime = t, location = new_york))

In [70]:
np.array(local_sun.az)

array([ 65.0880044 ,  65.89090363,  66.6885197 ,  67.48117328,
        68.26918781,  69.05288957,  69.83260797,  70.60867575,
        71.38142929,  72.15120892,  72.9183593 ,  73.68322984,
        74.44617512,  75.20755543,  75.96773725,  76.72709391,
        77.48600615,  78.24486287,  79.00406185,  79.76401054,
        80.52512692,  81.28784046,  82.05259308,  82.81984024,
        83.59005209,  84.36371471,  85.14133143,  85.92342428,
        86.71053552,  87.50322931,  88.30209348,  89.10774144,
        89.92081428,  90.74198297,  91.57195078,  92.41145583,
        93.26127391,  94.12222145,  94.99515873,  95.88099341,
        96.78068417,  97.6952448 ,  98.62574846,  99.57333228,
       100.5392023 , 101.5246387 , 102.53100143, 103.55973608,
       104.61238015, 105.69056957, 106.79604554, 107.93066151,
       109.09639039, 110.29533167, 111.52971851, 112.80192442,
       114.11446938, 115.47002502, 116.87141839, 118.32163375,
       119.82381179, 121.38124531, 122.99737029, 124.67

In [69]:
np.array(local_sun.alt)

array([ 1.72118445,  2.58138163,  3.44697986,  4.31777717,  5.19357586,
        6.07418221,  6.95940636,  7.84906202,  8.74296633,  9.64093957,
       10.54280497, 11.44838848, 12.35751853, 13.2700258 , 14.18574296,
       15.10450445, 16.0261462 , 16.95050535, 17.87742004, 18.80672907,
       19.73827164, 20.67188702, 21.60741429, 22.54469194, 23.48355758,
       24.42384757, 25.36539662, 26.30803737, 27.25160003, 28.19591186,
       29.14079671, 30.0860745 , 31.03156069, 31.97706562, 32.92239395,
       33.86734392, 34.8117066 , 35.75526512, 36.69779376, 37.639057  ,
       38.57880851, 39.51678998, 40.45272994, 41.38634235, 42.3173252 ,
       43.24535883, 44.17010422, 45.09120102, 46.00826546, 46.920888  ,
       47.82863082, 48.73102501, 49.62756752, 50.51771785, 51.40089438,
       52.27647045, 53.14377005, 54.00206317, 54.85056085, 55.68840976,
       56.51468664, 57.32839228, 58.12844538, 58.91367628, 59.68282067,
       60.43451355, 61.16728368, 61.87954877, 62.569612  , 63.23

In [72]:
df_m['sun_altitude'] = np.array(local_sun.alt)

In [75]:
df_m

,Date / Time,Air Temp at Surface [degC],Relative Humidity [percent],Avg Wind Speed [m/s],Wind Direction [degrees],Solar Flux [W/m^2],sun_altitude
0,2021-07-24 06:00:00,21.3,66.5,0.9,348,10,1.721184
1,2021-07-24 06:05:00,21.4,66.1,1.1,345,12,2.581382
2,2021-07-24 06:10:00,21.4,66.5,1.3,4,14,3.446980
3,2021-07-24 06:15:00,21.5,65.4,1.3,5,17,4.317777
4,2021-07-24 06:20:00,21.5,65.0,1.5,346,19,5.193576
...,...,...,...,...,...,...,...
164,2021-07-24 19:40:00,25.0,46.8,2.2,168,21,5.908307
165,2021-07-24 19:45:00,24.9,47.3,3.0,167,18,5.026643
166,2021-07-24 19:50:00,24.8,48.0,2.4,184,19,4.149777
167,2021-07-24 19:55:00,24.8,48.0,2.0,182,18,3.277902


In [84]:
df_m = df_m.rename(columns = {df_m.columns[1] : "air_temperature_at_surface_manhattan [degC]",
                                  df_m.columns[2] : "relative_humidity_manhattan [percent]",
                                  df_m.columns[3] : "avg_wind_speed_manhattan [m/s]",
                                  df_m.columns[4] : "wind_direction_manhattan [degrees]",
                                  df_m.columns[5] : "solar_flux_manhattan [W/m^2]"})

In [85]:
df_b = df_b.rename(columns = {df_b.columns[1] : "air_temperature_at_surface_bronx [degC]",
                      df_b.columns[2] : "relative_humidity_bronx [percent]",
                      df_b.columns[3] : "avg_wind_speed_bronx [m/s]",
                      df_b.columns[4] : "wind_direction_bronx [degrees]",
                      df_b.columns[5] : "solar_flux_bronx [W/m^2]"})

In [86]:
pd.concat((df_m, df_b.drop(columns = [df_b.columns[0]])), axis = 1)

,Date / Time,air_temperature_at_surface_manhattan [degC],relative_humidity_manhattan [percent],avg_wind_speed_manhattan [m/s],wind_direction_manhattan [degrees],solar_flux_manhattan [W/m^2],sun_altitude,air_temperature_at_surface_bronx [degC],relative_humidity_bronx [percent],avg_wind_speed_bronx [m/s],wind_direction_bronx [degrees],solar_flux_bronx [W/m^2]
0,2021-07-24 06:00:00,21.3,66.5,0.9,348,10,1.721184,19.3,88.2,0.8,335,12
1,2021-07-24 06:05:00,21.4,66.1,1.1,345,12,2.581382,19.4,87.9,0.8,329,18
2,2021-07-24 06:10:00,21.4,66.5,1.3,4,14,3.446980,19.3,87.6,0.7,321,25
3,2021-07-24 06:15:00,21.5,65.4,1.3,5,17,4.317777,19.4,87.4,0.5,307,33
4,2021-07-24 06:20:00,21.5,65.0,1.5,346,19,5.193576,19.4,87.0,0.2,301,42
...,...,...,...,...,...,...,...,...,...,...,...,...
164,2021-07-24 19:40:00,25.0,46.8,2.2,168,21,5.908307,24.9,49.0,3.5,184,24
165,2021-07-24 19:45:00,24.9,47.3,3.0,167,18,5.026643,24.8,49.0,3.3,173,19
166,2021-07-24 19:50:00,24.8,48.0,2.4,184,19,4.149777,24.9,48.7,3.8,168,17
167,2021-07-24 19:55:00,24.8,48.0,2.0,182,18,3.277902,24.9,47.3,4.1,171,16


In [80]:
for col in df_m.columns:
    print(col)

Date / Time
Air Temp at Surface [degC]
Relative Humidity [percent]
Avg Wind Speed [m/s]
Wind Direction [degrees]
Solar Flux [W/m^2]
sun_altitude
